In [ ]:
# Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
from matplotlib.ticker import PercentFormatter
from matplotlib.ticker import FuncFormatter
from matplotlib.patches import FancyBboxPatch
from matplotlib.lines import Line2D


import warnings
warnings.filterwarnings("ignore")   

In [ ]:
# Load the all dataset

In [ ]:
gold_loans = pd.read_csv('/data/gold_loans.csv')
repayments = pd.read_csv('/data/repayments.csv')
gold_rates = pd.read_csv('/data/gold_rates.csv')
branches = pd.read_csv('/data/branches.csv')
customers = pd.read_csv('/data/customers.csv')

In [ ]:
# Copy all the original dataset

In [ ]:
df_gl = gold_loans.copy()
df_rp = repayments.copy()
df_gr = gold_rates.copy()
df_br = branches.copy()
df_cs = customers.copy()

In [ ]:
# Solve the Questions

In [ ]:
# 1. Size of the loan extract : How many rows are in gold_loans exactly as received (before any cleaning)?

In [ ]:
print('Total Number of Rows in Gold Loans:', df_gl.shape[0])

In [ ]:
# 2. Size of the repayments table : How many rows are in repayments?

In [ ]:
print('Total Number of Rows in Repayments:', df_rp.shape[0])

In [ ]:
# 3. Branch network : How many branches does Suvarna Finance have?

In [ ]:
print('Total Number of Branches in Suvarna Finance:', df_br['branch_id'].nunique())

In [ ]:
# 4. States covered : In how many different states does the company have branches?

In [ ]:
print('Total Number of Different States Covered by Suvarna Finance:', df_br['state'].nunique())

In [ ]:
# 5. Disbursal period : Return one row with the earliest and latest disbursal dates as first_disbursal and last_disbursal.

In [ ]:
answer = df_gl['disbursal_date'].agg({
    'first_disbursal' : 'min',
    'last_disbursal' : 'max'
}).to_frame(name='disbursal_period').T

answer

In [ ]:
# 6. Unique loans : How many distinct loan_id values are there?

In [ ]:
print('Total Number of Unique Loans:', df_gl['loan_id'].nunique())

In [ ]:
# 7. Duplicate rows : How many exact duplicate rows must be removed from gold_loans?

In [ ]:
print('Total Number of Exact Duplicate Rows in Gold Loans:', df_gl.duplicated().sum())

In [ ]:
# Drop duplicated rows

df_gl.drop_duplicates(inplace=True)

# After Removal of duplicates again check duplicated records
print('Total Duplicate Rows in Gold Loans After Removal:', df_gl.duplicated().sum())

In [ ]:
# 8. Inconsistent status values : In how many different ways is status written in the raw data (count distinct raw values, letter case included)?

In [ ]:
print('Total Number of Distinct Status Values in Raw Data:', df_gl['status'].nunique())

In [ ]:
# 9. Loans by status : After removing duplicates and converting status to lower case, return status and loans (the number of loans) for each status.

In [ ]:
# Check Inconsistent Status Values

print('Unique Status Values Before Standardization:', df_gl['status'].unique())

In [ ]:
# Standardize Inconsistent Status Values

df_gl['status'] = df_gl['status'].str.strip().str.lower()

print('Unique Status Values After Standardization:', df_gl['status'].unique())

In [ ]:
answer = df_gl['status'].value_counts().rename('loans')
answer

In [ ]:
# 10. Purity formats : How many different raw values does the purity column contain?

In [ ]:
print('Total Number of Different Raw Values in Purity Column:', df_gl['purity'].nunique())

In [ ]:
# 11. Loans by karat : After removing duplicates and converting purity to a karat number (916 = 22, 833 = 20, 750 = 18, 999 = 24), return karat and loans for each karat.

In [ ]:
# Check Inconsistent Purity Values

print('Unique Purity Values Before Standardization:', df_gl['purity'].unique())

In [ ]:
# Standardize Inconsistent Purity Values into one scale (karat)

df_gl['purity'] = (
    df_gl['purity']
    .replace(r'\s|K|T', '', regex=True)
    .replace({'916':22,'833':20,'750':18,'999':24})
    .astype(int)
)

print('Unique Status Values After Standardization:', df_gl['purity'].unique())


In [ ]:
answer = df_gl['purity'].value_counts().rename_axis('karat').rename('loans')
answer

In [ ]:
# 12. Missing gold weight : After removing duplicates, how many loans have no gold weight recorded?

In [ ]:
answer = df_gl['gold_weight_gm'].isna().sum()
answer

In [ ]:
# 13. Invalid principal : After removing duplicates, how many loans have a principal of 0 or less?

In [ ]:
answer = len(df_gl[df_gl['principal'] <= 0])
answer

In [ ]:
# 14. Active loans : How many active loans are there after removing duplicates (status 'active' in any letter case)?

In [ ]:
answer = len(df_gl[df_gl['status'] == 'active'])
answer

In [ ]:
# 15. Loans that cannot be assessed : How many active loans cannot be assessed because the gold weight is missing or the principal is 0 or less?

In [ ]:
answer = len(df_gl[(df_gl['status']=='active') & ((df_gl['principal'] <= 0) | (df_gl['gold_weight_gm'].isnull()))])
answer

In [ ]:
# 16. Gold rate history : How many daily rates are in gold_rates?

In [ ]:
answer = len(df_gr)
answer

In [ ]:
# 17. 2026 peak gold rate : What was the highest 24K gold rate (₹ per gram) during 2026?

In [ ]:
# Convert Datatype of rate_date OBJECT to DATETIME
df_gr['rate_date'] = pd.to_datetime(df_gr['rate_date'], errors='coerce')

answer = df_gr.loc[
    df_gr['rate_date'].dt.year == 2026,
    'rate_24k_per_gram'
].max()

answer

In [ ]:
# 18. Date of the peak : Return one row with peak_date: the date of the highest 24K rate in 2026.

In [ ]:
answer = df_gr[
    (df_gr['rate_date'].dt.year == 2026)
    & 
    (df_gr['rate_24k_per_gram'] == df_gr['rate_24k_per_gram'].max())
]['rate_date'].reset_index(name='peak_date')

answer

In [ ]:
# 19. Rate on the assessment date : Which 24K rate (₹ per gram) applies on 15 October 2026?

In [ ]:
df_gr[df_gr['rate_date'] == '2026-10-15']

In [ ]:
# 20. Fall from the peak : By what percentage has the 24K rate fallen from its 2026 peak to the assessment date? Round to 2 decimals.

In [ ]:
peak_rate = df_gr[df_gr['rate_date'].dt.year == 2026]['rate_24k_per_gram'].max()

assessment_rate = df_gr.loc[df_gr['rate_date'] == '2026-10-15', 'rate_24k_per_gram'].iloc[0]

fall_percentage = round((peak_rate - assessment_rate) / peak_rate * 100, 2)

fall_percentage

In [ ]:
# 21. City names in the branch master : How many distinct city names appear in branches? (Old names such as Bangalore count separately.)

In [ ]:
answer = df_br['city'].nunique()
answer

In [ ]:
# 22. Loans by scheme : After removing duplicates, return scheme and loans for each scheme.

In [ ]:
answer = df_gl.groupby('scheme')['loan_id'].count().reset_index(name='loans')
answer

In [ ]:
# 23. Total disbursed : After removing duplicates, what is the total principal disbursed across all loans with a principal above 0 (₹)?

In [ ]:
df_gl[df_gl['principal'] > 0]['principal'].sum()

In [ ]:
# 24. Average active loan : What is the average principal (₹, rounded to 0 decimals) of assessable active loans?

In [ ]:
round(df_gl[df_gl['status'] == 'active']['principal'].mean(), 0)

In [ ]:
# 25. Customer base : How many customers are in customers?

In [ ]:
len(df_cs)

In [ ]:
# 26. Average customer age : What is the average customer age? Round to 1 decimal.

In [ ]:
df_cs['age'].mean().round(1)

In [ ]:
# 27. Loans disbursed in 2026 : After removing duplicates, how many loans were disbursed in 2026?

In [ ]:
# Convert Datatype of rate_date OBJECT to DATETIME
df_gl['disbursal_date'] = pd.to_datetime(df_gl['disbursal_date'], errors='coerce')

len(df_gl[df_gl['disbursal_date'].dt.year == 2026])

In [ ]:
# 28. Interest rate by scheme : After removing duplicates, return scheme and avg_rate (average interest_rate_pct, rounded to 2 decimals).

In [ ]:
answer = df_gl.groupby('scheme')['interest_rate_pct'].mean().round(2).reset_index(name='avg_rate')
answer

In [ ]:
# 29. Principal repayments : How many repayments have payment_type = 'principal'?

In [ ]:
len(df_rp[df_rp['payment_type'] == 'principal'])

In [ ]:
# 30. Interest collected : What is the total amount received as interest (₹)?

In [ ]:
df_rp.head(5)

In [ ]:
df_rp[df_rp['payment_type'] == 'interest']['amount'].sum()

In [ ]:
# ****************************************** *INTERMEDIATE QUESTIONS* *****************************************

In [ ]:
# 31. Outstanding principal : What is the total outstanding principal (₹) of all assessable active loans? Outstanding = principal − principal repayments.

In [ ]:
assessment_date = pd.Timestamp('2026-10-15')
rate_asof = df_gr[df_gr['rate_date'] <= assessment_date].sort_values('rate_date')['rate_24k_per_gram'].iloc[-1]

In [ ]:
repaid = df_rp[df_rp['payment_type'] == 'principal'].groupby('loan_id')['amount'].sum()
repaid.head(2)

In [ ]:
book = df_gl[(df_gl['status'] == 'active') & (df_gl['principal'] > 0) & (df_gl['gold_weight_gm'].notna())]
book['outstanding'] = book['principal'] - book['loan_id'].map(repaid).fillna(0)
book['collateral'] = book['gold_weight_gm'] * book['purity'] / 24 * rate_asof
book['current_ltv'] = book['outstanding'] /  book['collateral']

book.head()

In [ ]:
book['outstanding'].sum()

In [ ]:
# 32. Average current LTV : What is the average current LTV (%) of assessable active loans? Round to 2 decimals.

In [ ]:
(100 * book['current_ltv'].mean()).round(2)

In [ ]:
# 33. Loans in breach : How many assessable active loans are above the 75% LTV cap?

In [ ]:
breach = book[book['current_ltv'] > 0.75]
breach.head(3)

In [ ]:
len(book[book['current_ltv'] > 0.75])

In [ ]:
# 34. Money in breach : What is the outstanding principal (₹) of loans above 75% LTV?

In [ ]:
breach['outstanding'].sum()

In [ ]:
# 35. Share of the book in breach : What percentage of assessable active loans are in breach? Round to 2 decimals.

In [ ]:
round(100 * len(breach)/len(book), 2)

In [ ]:
# 36. Breaches by scheme : Return scheme, breach_loans and active_loans (assessable active loans) for each scheme.

In [ ]:
answer = book.groupby('scheme').agg(
    breach_loans = ('current_ltv', lambda x : (x > 0.75).sum()),
    active_loans = ('loan_id', 'count')
).reset_index()

answer

In [ ]:
# 37. Breach exposure by state : Return state and breach_outstanding (₹, rounded to 0) for every state with breaches, highest first.

In [ ]:
answer = breach.merge(df_br, on='branch_id').groupby('state')['outstanding'].sum().round(0).sort_values(ascending=False).reset_index(name='breach_outstanding')
answer

In [ ]:
# 38. Breaches by disbursal month : For breaching loans, return month (text 'YYYY-MM' of the disbursal date) and breach_loans, in month order.

In [ ]:
breach['month'] = breach['disbursal_date'].dt.strftime('%Y-%m')
answer = breach.groupby('month').size().reset_index(name='breach_loans')
answer

In [ ]:
# 39. Top 10 branches : Return the top 10 branches by breached outstanding: branch_id, branch_name, breach_outstanding (₹, rounded to 0), highest first.

In [ ]:
answer = breach.merge(df_br, on='branch_id').groupby(['branch_id', 'branch_name'])['outstanding'].sum().round(0).sort_values(ascending=False).head(10).reset_index(name='breach_outstanding')

answer

In [ ]:
# 40. Express loans in breach : What percentage of assessable active Express loans are in breach? Round to 1 decimal.

In [ ]:
express = book[book['scheme'] == 'Express']

answer = round(100 * (express['current_ltv'] > 0.75).sum() / len(express), 1)
answer

In [ ]:
# 41. Book by region : Return region, active_loans and outstanding (₹, rounded to 0) for assessable active loans in each region.

In [ ]:
answer = book.merge(df_br, on='branch_id').groupby('region').agg(
    active_loans=('loan_id', 'count'),
    outstanding=('outstanding', 'sum')
).round(0).reset_index()

answer

In [ ]:
# 42. LTV at disbursal near the peak : For loans disbursed from 1 July to 30 September 2026 (after removing duplicates), return scheme and avg_ltv_at_disbursal (rounded to 2).

In [ ]:
# Filter loans disbursed between 1 July and 30 September 2026
q42 = df_gl[
    df_gl['disbursal_date'].between('2026-07-01', '2026-09-30')
].copy()

# Average LTV at disbursal by scheme
answer = (
    q42.groupby('scheme')['ltv_at_disbursal_pct']
    .mean()
    .round(2)
    .reset_index(name='avg_ltv_at_disbursal')
)

answer

In [ ]:
# 43. Overdue active loans : How many active loans (after removing duplicates) are past their maturity date (disbursal date + tenure months) on 15 October 2026?

In [ ]:
assessment_date = pd.Timestamp('2026-10-15')

active_loans = df_gl[df_gl['status'] == 'active'].copy()

active_loans['maturity_date'] = active_loans.apply(
    lambda x: x['disbursal_date'] + pd.DateOffset(months=x['tenure_months']),
    axis=1
)

answer = len(
    active_loans[active_loans['maturity_date'] < assessment_date]
)

answer

In [ ]:
# 44. Customers affected : How many distinct customers have at least one loan in breach?

In [ ]:
answer = breach['customer_id'].nunique()

answer

In [ ]:
# 45. Breaches by occupation : Return occupation and breach_loans, highest first.

In [ ]:
answer = breach.merge(df_cs, on='customer_id').groupby('occupation').agg(
    breach_loans=('loan_id', 'count')
).reset_index().sort_values('breach_loans', ascending=False)

answer

In [ ]:
# ******************************************** *ADVANCED QUESTIONS* *******************************************

In [ ]:
# 46. A further 5% fall : If the gold rate falls a further 5% from the assessment-date rate, how many assessable active loans would be above 75% LTV?

In [ ]:
book['deduction_5%'] = book['collateral'] * 0.95
ltv_after_fall = book['outstanding'] / book['deduction_5%']

(ltv_after_fall > 0.75).sum()

In [ ]:
# 47. Margin call amount : How much principal (₹, rounded to 0) must be repaid in total to bring every breaching loan back to exactly 75% LTV?

In [ ]:
book['breach'] = book['current_ltv'] > 0.75
breaching = book.loc[book['breach']]

In [ ]:
(breaching['outstanding'] - 0.75 * breaching['collateral']).sum().round(2)

In [ ]:
# 48. Critical breaches : How many assessable active loans are above 80% LTV (critical)?

In [ ]:
(book['current_ltv'] > 0.80).sum()

In [ ]:
# 49. Worst branch in each region : For each region, return the branch with the highest breach rate (% of its assessable active loans in breach): region, branch_name, breach_rate_pct (rounded to 2).

In [ ]:
branch_breach = book.merge(df_br, on='branch_id').groupby(['region', 'branch_name']).agg(
    breaching = ('breach', 'sum'),
    df_gl = ('loan_id', 'size')
).reset_index()

branch_breach['breach_rate_pct'] = round(branch_breach['breaching'] / branch_breach['df_gl'] * 100, 2)

In [ ]:
answer = branch_breach[['region', 'branch_name', 'breach_rate_pct']]

answer = answer.sort_values(['region','breach_rate_pct','branch_name'],ascending=[True,False,True]).groupby('region').head(1)

answer

In [ ]:
# 50. Cumulative exposure by month : For breaching loans disbursed in 2026, return month ('YYYY-MM'), breach_outstanding (rounded to 0) and cumulative_outstanding (running total, rounded to 0), in month order.

In [ ]:
cumulative_outstanding = breaching.loc[breaching['disbursal_date'].dt.year == 2026].assign(
    month=lambda x : x['disbursal_date'].dt.strftime('%Y-%m')
).groupby('month', as_index=False)['outstanding'].sum().sort_values('month')

In [ ]:
cumulative_outstanding['breach_outstanding'] = cumulative_outstanding['outstanding'].round(0)
cumulative_outstanding['cumulative_outstanding'] = cumulative_outstanding['breach_outstanding'].cumsum()

In [ ]:
answer = cumulative_outstanding[['month', 'breach_outstanding', 'cumulative_outstanding']]
answer

In [ ]:
# VISUALIZATION USING MATPLOLIB & SEABORN

In [ ]:
# Chart 1: Scheme-wise LTV Breach Rate analysis

- This chart compares the percentage of assessable active loans exceeding the RBI's 75% LTV threshold across Agri, Standard, and Express loan schemes.

- The objective is to identify which loan scheme has the highest concentration of LTV breaches and requires immediate risk-control attention.

In [ ]:
# Prepare scheme-wise breach metrics
scheme_chart = (
    book.assign(is_breach=book['current_ltv'] > 0.75)
    .groupby('scheme')
    .agg(
        active_loans=('loan_id', 'count'),
        breached_loans=('is_breach', 'sum')
    )
    .reset_index()
)

scheme_chart['breach_rate'] = (
    scheme_chart['breached_loans'] /
    scheme_chart['active_loans'] * 100
)

scheme_chart = scheme_chart.sort_values('breach_rate')

# Configure chart appearance
sns.set_theme(style="whitegrid", font="DejaVu Sans")

fig, ax = plt.subplots(figsize=(12, 5.8))

colors = {
    'Agri': '#94A3B8',
    'Standard': '#64748B',
    'Express': '#DC5353'
}

bars = ax.barh(
    scheme_chart['scheme'],
    scheme_chart['breach_rate'],
    color=[colors[s] for s in scheme_chart['scheme']],
    height=0.52,
    edgecolor='white',
    linewidth=1.2
)

# Add percentage and loan-count labels
for bar, (_, row) in zip(bars, scheme_chart.iterrows()):
    ax.text(
        bar.get_width() + 1.5,
        bar.get_y() + bar.get_height() / 2,
        f"{row['breach_rate']:.2f}%  |  {int(row['breached_loans']):,} / {int(row['active_loans']):,}",
        va='center',
        fontsize=10,
        fontweight='bold',
        color='#263445'
    )

# Style axes and title
ax.set_title(
    "LTV BREACH RATE BY LOAN SCHEME",
    fontsize=16,
    fontweight='bold',
    color='#1E293B',
    loc='left',
    pad=22
)

ax.text(
    0, 1.02,
    "Share of assessable active loans exceeding the 75% LTV threshold",
    transform=ax.transAxes,
    fontsize=10,
    color='#64748B'
)

ax.set_xlabel("Loans Above 75% LTV (%)", fontsize=10, labelpad=12)
ax.set_ylabel("Loan Scheme", fontsize=10, labelpad=12)

ax.set_xlim(0, 115)
ax.xaxis.set_major_formatter(PercentFormatter(xmax=100))

ax.grid(axis='x', linestyle='--', alpha=0.25)
ax.grid(axis='y', visible=False)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)

ax.tick_params(axis='both', labelsize=10, length=0)

fig.text(
    0.12, 0.015,
    "Source: Suvarna Finance loan book | Assessment date: 15 October 2026",
    fontsize=9,
    color='#64748B'
)

plt.tight_layout(rect=[0, 0.05, 1, 0.96])
plt.show()

In [ ]:
**Key Findings:**

- **Express:** 97.05% breach rate (1,548 out of 1,595 assessable active loans), indicating an exceptionally high concentration of LTV risk.
- **Standard:** 6.09% breach rate (952 out of 15,623 loans).
- **Agri:** 0.13% breach rate (2 out of 1,557 loans).

**Business Insight:** The Express scheme is the dominant source of LTV breach concentration. It requires immediate review of its high-LTV origination policy and stronger collateral-risk controls.

In [ ]:
# Chart 2: Monthly LTV Breach Trend — June to October 2026

This analysis examines the monthly distribution of LTV breaches across loans disbursed between June and October 2026. It compares the number of breached loans with their associated outstanding principal exposure to identify periods of elevated portfolio risk.

The analysis helps identify recent disbursal cohorts contributing significantly to the overall LTV breach exposure.

In [ ]:
# 1. Prepare monthly breach metrics
monthly_chart = (
    book.assign(
        disbursal_month=pd.to_datetime(book['disbursal_date'])
        .dt.to_period('M').astype(str),
        is_breach=book['current_ltv'] > 0.75
    )
    .query("is_breach == True")
    .groupby('disbursal_month')
    .agg(
        breached_loans=('loan_id', 'count'),
        breached_outstanding=('outstanding', 'sum')
    )
    .reset_index()
)

# 2. Arrange months chronologically
months = ['2026-06', '2026-07', '2026-08', '2026-09', '2026-10']

monthly_chart = (
    monthly_chart.set_index('disbursal_month')
    .reindex(months, fill_value=0)
    .rename_axis('disbursal_month')
    .reset_index()
)

monthly_chart['outstanding_cr'] = (
    monthly_chart['breached_outstanding'] / 10000000
)

# 3. Configure professional chart appearance
sns.set_theme(style="whitegrid", font="DejaVu Sans")

fig, ax1 = plt.subplots(figsize=(12, 5.8))
fig.patch.set_facecolor('white')
ax1.set_facecolor('white')

x = np.arange(len(monthly_chart))

# 4. Draw monthly breached loan count bars
bars = ax1.bar(
    x,
    monthly_chart['breached_loans'],
    width=0.55,
    color='#52769F',
    edgecolor='white',
    linewidth=1.2,
    zorder=3
)

# 5. Add loan count labels inside the bars
for bar, value in zip(bars, monthly_chart['breached_loans']):
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() * 0.72,
        f'{int(value):,}',
        ha='center',
        va='center',
        fontsize=11,
        fontweight='bold',
        color='white'
    )

# 6. Draw outstanding exposure line on secondary axis
ax2 = ax1.twinx()

ax2.plot(
    x,
    monthly_chart['outstanding_cr'],
    color='#DC5353',
    linewidth=2.8,
    marker='o',
    markersize=8,
    markerfacecolor='white',
    markeredgecolor='#DC5353',
    markeredgewidth=2.2,
    zorder=5
)

# 7. Add outstanding amount labels with separate backgrounds
for xpos, value in zip(x, monthly_chart['outstanding_cr']):
    ax2.annotate(
        f'₹{value:.2f} Cr',
        (xpos, value),
        xytext=(0, 17),
        textcoords='offset points',
        ha='center',
        va='bottom',
        fontsize=9,
        fontweight='bold',
        color='#C34B55',
        bbox=dict(
            boxstyle='round,pad=0.35',
            facecolor='white',
            edgecolor='#F1D5D7',
            alpha=0.95
        ),
        zorder=10
    )

# 8. Configure X-axis
ax1.set_xticks(x)
ax1.set_xticklabels(
    pd.to_datetime(monthly_chart['disbursal_month']).dt.strftime('%b %Y'),
    fontsize=10
)

# 9. Configure Y-axes
ax1.set_ylabel(
    'Number of Breached Loans',
    fontsize=10,
    labelpad=12,
    color='#52769F'
)

ax2.set_ylabel(
    'Breached Outstanding (₹ Crore)',
    fontsize=10,
    labelpad=12,
    color='#DC5353'
)

ax1.set_xlabel('Disbursal Month', fontsize=10, labelpad=12)

ax1.set_ylim(
    0,
    monthly_chart['breached_loans'].max() * 1.25
)

ax2.set_ylim(
    0,
    monthly_chart['outstanding_cr'].max() * 1.35
)

# 10. Format axes and gridlines
ax1.yaxis.set_major_formatter(
    FuncFormatter(lambda value, _: f'{value:,.0f}')
)

ax2.yaxis.set_major_formatter(
    FuncFormatter(lambda value, _: f'₹{value:.0f}')
)

ax1.grid(axis='y', linestyle='--', alpha=0.25)
ax1.grid(axis='x', visible=False)
ax2.grid(False)

for axis in [ax1, ax2]:
    axis.spines['top'].set_visible(False)
    axis.spines['left'].set_visible(False)
    axis.spines['right'].set_visible(False)

ax1.tick_params(axis='both', labelsize=10, length=0)
ax2.tick_params(axis='y', labelsize=10, length=0)

# 11. Add chart title and subtitle
ax1.set_title(
    "MONTHLY LTV BREACH TREND",
    fontsize=16,
    fontweight='bold',
    color='#1E293B',
    loc='left',
    pad=42
)

ax1.text(
    0, 1.08,
    "Monthly breached loan count and associated outstanding exposure",
    transform=ax1.transAxes,
    fontsize=10,
    color='#64748B'
)

# 12. Add chart legend
legend_items = [
    plt.Rectangle(
        (0, 0), 1, 1,
        color='#52769F',
        label='Breached loan count'
    ),
    Line2D(
        [0], [0],
        color='#DC5353',
        marker='o',
        linewidth=2.8,
        label='Breached outstanding'
    )
]

ax1.legend(
    handles=legend_items,
    loc='upper left',
    bbox_to_anchor=(0, 1.02),
    frameon=False,
    ncol=2,
    fontsize=9
)

# 13. Add source and assessment date
fig.text(
    0.12, 0.015,
    "Source: Suvarna Finance loan book | Assessment date: 15 October 2026",
    fontsize=9,
    color='#64748B'
)

plt.tight_layout(rect=[0, 0.05, 1, 0.96])
plt.show()

In [ ]:
**Key Findings:**

- **September 2026** recorded the highest number of breached loans, with 814 loans.
- September also recorded the highest breached outstanding exposure of **₹212.67 crore**.
- August and September together accounted for 1,575 breached loans.
- October figures cover loans disbursed only up to 15 October 2026.

**Business Insight:**

The concentration of LTV breaches in the August and September disbursal cohorts indicates elevated risk among recent originations. Management should review underwriting practices, initial LTV levels and collateral valuation controls for these cohorts.

In [ ]:
# Chart 3: State-wise LTV Breach Outstanding Exposure

This analysis compares the outstanding principal associated with LTV-breached loans across all 11 states. States are ranked in descending order of breached outstanding exposure to identify the geographical concentration of portfolio risk.

The objective is to identify priority states for focused risk monitoring and margin-call activities.

In [ ]:
# 1. Prepare state-wise breached outstanding data
state_chart = pd.DataFrame({
    'state': [
        'Tamil Nadu',
        'Telangana',
        'Kerala',
        'Maharashtra',
        'Gujarat',
        'Karnataka',
        'Delhi',
        'Andhra Pradesh',
        'Uttar Pradesh',
        'Rajasthan',
        'Haryana'
    ],
    'breached_outstanding': [
        312878900,
        144945600,
        44414200,
        34299900,
        30156400,
        26839100,
        18362800,
        15887500,
        8051800,
        6369700,
        5899200
    ]
})

# 2. Convert outstanding amount into crore
state_chart['outstanding_cr'] = (
    state_chart['breached_outstanding'] / 10000000
)

# 3. Calculate contribution to total breached exposure
total_exposure = state_chart['breached_outstanding'].sum()

state_chart['exposure_share'] = (
    state_chart['breached_outstanding'] / total_exposure * 100
)

state_chart = state_chart.sort_values(
    'outstanding_cr',
    ascending=True
)

# 4. Configure professional chart appearance
sns.set_theme(style="whitegrid", font="DejaVu Sans")

fig, ax = plt.subplots(figsize=(12, 7))

fig.patch.set_facecolor('white')
ax.set_facecolor('white')

colors = [
    '#DC5353' if state in ['Tamil Nadu', 'Telangana']
    else '#52769F'
    for state in state_chart['state']
]

# 5. Draw horizontal bars
bars = ax.barh(
    state_chart['state'],
    state_chart['outstanding_cr'],
    color=colors,
    height=0.62,
    edgecolor='white',
    linewidth=1.2
)

# 6. Add amount and exposure-share labels
for bar, (_, row) in zip(bars, state_chart.iterrows()):
    ax.text(
        bar.get_width() + 0.35,
        bar.get_y() + bar.get_height() / 2,
        f"₹{row['outstanding_cr']:.2f} Cr  |  {row['exposure_share']:.1f}%",
        va='center',
        fontsize=9,
        fontweight='bold',
        color='#263445'
    )

# 7. Style chart title and subtitle
ax.set_title(
    "STATE-WISE BREACHED OUTSTANDING EXPOSURE",
    fontsize=16,
    fontweight='bold',
    color='#1E293B',
    loc='left',
    pad=24
)

ax.text(
    0, 1.015,
    "Geographical distribution of outstanding principal above the 75% LTV threshold",
    transform=ax.transAxes,
    fontsize=10,
    color='#64748B'
)

# 8. Configure axes and gridlines
ax.set_xlabel(
    "Breached Outstanding (₹ Crore)",
    fontsize=10,
    labelpad=12
)

ax.set_ylabel("State", fontsize=10, labelpad=12)

ax.set_xlim(0, 37)

ax.xaxis.set_major_formatter(
    FuncFormatter(lambda value, _: f'₹{value:.0f} Cr')
)

ax.grid(axis='x', linestyle='--', alpha=0.25)
ax.grid(axis='y', visible=False)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)

ax.tick_params(axis='both', labelsize=9, length=0)

# 9. Add source and assessment date
fig.text(
    0.12, 0.015,
    "Source: Suvarna Finance loan book | Assessment date: 15 October 2026",
    fontsize=9,
    color='#64748B'
)

plt.tight_layout(rect=[0, 0.05, 1, 0.96])
plt.show()

In [ ]:
**Key Findings:**

- **Tamil Nadu** has the highest breached outstanding exposure at ₹31.29 crore.
- **Telangana** ranks second with ₹14.49 crore.
- Tamil Nadu and Telangana together account for approximately **70.64%** of total breached outstanding exposure.
- Kerala ranks third with ₹4.44 crore in breached outstanding exposure.

**Business Insight:**

LTV breach exposure is heavily concentrated in Tamil Nadu and Telangana. Management should prioritise these two states for margin-call execution, branch-level monitoring and focused recovery planning.

In [ ]:
# Chart 4: Top 10 Branches by LTV Breached Outstanding

This analysis identifies the ten branches with the highest outstanding principal exposure from loans exceeding the 75% LTV threshold.

The branches are ranked by breached outstanding amount to identify priority locations for targeted risk monitoring, margin-call execution and recovery planning.

In [ ]:
# Prepare top 10 branch-wise breached outstanding data
branch_chart = pd.DataFrame({
    'branch_id': [
        'BR011', 'BR008', 'BR001', 'BR009', 'BR004',
        'BR006', 'BR010', 'BR007', 'BR013', 'BR012'
    ],
    'branch_name': [
        'Hyderabad Bazaar',
        'Madurai Gandhi Road',
        'Chennai Main Road',
        'Salem Main Road',
        'Chennai Nagar',
        'Coimbatore Main Road',
        'Hyderabad Station Road',
        'Madurai Bus Stand',
        'Warangal Nagar',
        'Hyderabad Station Road'
    ],
    'breached_outstanding': [
        44705400, 40324500, 39021100, 37566100, 37233300,
        34647900, 34345400, 33907400, 33157300, 32737500
    ]
})

# Convert outstanding amount into crore
branch_chart['outstanding_cr'] = (
    branch_chart['breached_outstanding'] / 10000000
)

# Create branch labels with IDs
branch_chart['branch_label'] = (
    branch_chart['branch_id'] + ' | ' + branch_chart['branch_name']
)

# Sort branches for horizontal bar chart
branch_chart = branch_chart.sort_values(
    'outstanding_cr',
    ascending=True
)

# Configure chart appearance
sns.set_theme(style="whitegrid", font="DejaVu Sans")

fig, ax = plt.subplots(figsize=(12, 6.5))

fig.patch.set_facecolor('white')
ax.set_facecolor('white')

# Highlight the top three branches
colors = [
    '#DC5353' if rank >= 7 else '#52769F'
    for rank in range(len(branch_chart))
]

# Draw horizontal bars
bars = ax.barh(
    branch_chart['branch_label'],
    branch_chart['outstanding_cr'],
    color=colors,
    height=0.62,
    edgecolor='white',
    linewidth=1.2
)

# Add outstanding amount labels
for bar, (_, row) in zip(bars, branch_chart.iterrows()):
    ax.text(
        bar.get_width() + 0.06,
        bar.get_y() + bar.get_height() / 2,
        f"₹{row['outstanding_cr']:.2f} Cr",
        va='center',
        fontsize=9,
        fontweight='bold',
        color='#263445'
    )

# Add chart title
ax.set_title(
    "TOP 10 BRANCHES BY BREACHED OUTSTANDING",
    fontsize=16,
    fontweight='bold',
    color='#1E293B',
    loc='left',
    pad=24
)

# Add subtitle
ax.text(
    0, 1.015,
    "Branches ranked by outstanding principal above the 75% LTV threshold",
    transform=ax.transAxes,
    fontsize=10,
    color='#64748B'
)

# Configure axes
ax.set_xlabel(
    "Breached Outstanding (₹ Crore)",
    fontsize=10,
    labelpad=12
)

ax.set_ylabel("Branch", fontsize=10, labelpad=12)

ax.set_xlim(0, 5.4)

ax.xaxis.set_major_formatter(
    FuncFormatter(lambda value, _: f'₹{value:.1f} Cr')
)

# Style gridlines and borders
ax.grid(axis='x', linestyle='--', alpha=0.25)
ax.grid(axis='y', visible=False)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)

ax.tick_params(axis='both', labelsize=9, length=0)

# Add source and assessment date
fig.text(
    0.12, 0.015,
    "Source: Suvarna Finance loan book | Assessment date: 15 October 2026",
    fontsize=9,
    color='#64748B'
)

plt.tight_layout(rect=[0, 0.05, 1, 0.96])
plt.show()

In [ ]:
**Key Findings:**

- **BR011 – Hyderabad Bazaar** recorded the highest breached outstanding exposure at **₹4.47 crore**.
- **BR008 – Madurai Gandhi Road** ranked second with **₹4.03 crore**.
- **BR001 – Chennai Main Road** ranked third with **₹3.90 crore**.
- Multiple high-exposure branches are concentrated in Tamil Nadu and Telangana.

**Business Insight:**

Management should prioritise the highest-exposure branches for borrower-level reviews, margin-call communication and recovery follow-ups. Branch-level monitoring should focus on both outstanding exposure and the number of affected borrowers before allocating recovery resources.